In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    recall_score, f1_score,
    confusion_matrix, roc_auc_score
)

from imblearn.over_sampling import SMOTE

#### <b>Préparation des données :  ##

In [15]:
df = pd.read_excel("churn_clean.xlsx")

In [16]:
y = df["Churn Value"]

cols_to_drop = [
    "CustomerID",
    "Churn Label",
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "Join Date",
    "Churn Date",
    "Latitude",
    "Longitude",
    "CLTV"
]

X = df.drop(columns=cols_to_drop, errors="ignore")

#### <b>Encodage et Split :  ##

In [17]:
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

#### <b>Équilibrage des classes :  ##

In [18]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

#### <b>Entraînement des modèles :  ##

In [19]:
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            eval_metric="logloss",
            random_state=42
        )
}

#### <b>Évaluation :  ##

In [20]:
results = []

for name, model in models.items():

    model.fit(X_train_smote, y_train_smote)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Modèle": name,
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

results_df = (
    pd.DataFrame(results)
    .set_index("Modèle")
    .round(3)
)

display(results_df)

c:\Users\pc\Desktop\Fil Rouge Customer Churn Analysis\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Recall,F1-Score,ROC-AUC
Modèle,,,
Logistic Regression,0.553,0.545,0.800
Random Forest,0.602,0.598,0.839
XGBoost,0.594,0.607,0.843


Les trois modèles obtiennent de bonnes performances, mais XGBoost est celui qui offre les meilleurs résultats globaux. Il présente le meilleur F1-Score et le meilleur ROC-AUC, ce qui signifie qu'il distingue mieux les clients qui vont résilier de ceux qui resteront. C'est pourquoi il a été retenu pour réaliser les prédictions de churn.

In [27]:
best_model_name = results_df["F1-Score"].idxmax()

best_model = models[best_model_name]

best_model.fit(
    X_train_smote,
    y_train_smote
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


#### <b>Probabilité de churn :  ##

In [22]:
df["Churn_Probability"] = (
    best_model.predict_proba(X)[:,1]
)

#### <b>Segmentation du risque :  ##

In [23]:
df["Risk_Level"] = pd.cut(
    df["Churn_Probability"],
    bins=[0,0.3,0.7,1],
    labels=["Faible","Moyen","Élevé"]
)

print(df["Risk_Level"].value_counts())

Risk_Level
Faible    4510
Élevé     1608
Moyen      559
Name: count, dtype: int64


#### <b>Clients actifs à risque

In [24]:
df["Customer_At_Risk"] = np.where(
    (df["Churn Value"] == 0) &
    (df["Risk_Level"] == "Élevé"),
    "Oui",
    "Non"
)

customers_at_risk = df[
    df["Customer_At_Risk"] == "Oui"
].copy()

print(f"Clients actifs à risque : {len(customers_at_risk)}")

Clients actifs à risque : 46


#### <b>Revenu  annuel à risque :  ##

In [25]:

df["Revenue_At_Risk"] = (
    df["Monthly Charges"] * 12 * df["Churn_Probability"]
)

revenu_risque = df.loc[
    df["Customer_At_Risk"] == "Oui",
    "Revenue_At_Risk"
].sum()

print(f"Revenu annuel à risque : {revenu_risque:,.0f} €")

Revenu annuel à risque : 36,483 €


#### <b>Export Power BI :  ##

In [26]:
# export_cols = [
#     "CustomerID",
#     "Churn_Probability",
#     "Risk_Level",
#     "Customer_At_Risk",
#     "Revenue_At_Risk"
# ]

# df[export_cols].to_excel(
#     "customers_with_predictions.xlsx",
#     index=False
# )